In [1]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import joblib
import os

# Local imports
from Scrapper import extract_article_text
from preprocess import clean_text
from logger_db import init_db, log_query

# Model paths
MODEL_DIR = "model_artifacts"
VECT_PATH = os.path.join(MODEL_DIR, "vectorizer.joblib")
MODEL_PATH = os.path.join(MODEL_DIR, "model.joblib")

# Initialize Flask app
app = Flask(__name__)
CORS(app)

# Load model and vectorizer
if not os.path.exists(VECT_PATH) or not os.path.exists(MODEL_PATH):
    raise FileNotFoundError("Model artifacts not found. Run train.py first.")

vectorizer = joblib.load(VECT_PATH)
model = joblib.load(MODEL_PATH)

# Init DB (optional logging)
init_db()

@app.route("/health", methods=["GET"])
def health():
    """Health check endpoint."""
    return jsonify({"status": "ok"}), 200


@app.route("/predict", methods=["POST"])
def predict():
    """
    Predict whether news is Real or Fake.
    Accepts JSON with either:
    - {"url": "https://example.com/article"}
    - {"text": "Some news article content..."}
    """
    data = request.get_json()

    url = data.get("url", "").strip()
    raw_text = data.get("text", "").strip()

    # Step 1: extract text from URL or use raw text
    if url:
        text = extract_article_text(url)
        if not text or len(text.strip()) < 100:
            return jsonify({"error": "Could not extract sufficient text from URL"}), 400
    elif raw_text:
        text = raw_text
    else:
        return jsonify({"error": "Provide either 'url' or 'text'"}), 400

    # Step 2: preprocess
    cleaned = clean_text(text)

    # Step 3: vectorize
    X = vectorizer.transform([cleaned])

    # Step 4: predict
    proba = model.predict_proba(X)[0]  # probabilities
    pred_index = proba.argmax()
    pred_class = model.classes_[pred_index]  # 0 or 1
    label = "Real" if pred_class == 1 else "Fake"
    confidence = float(proba[pred_index])

    # Step 5: optional logging
    try:
        log_query(url if url else "RAW_TEXT", label, confidence, len(text))
    except Exception as e:
        app.logger.warning("Logging failed: %s", e)

    return jsonify({
        "label": label,
        "confidence": round(confidence, 4)
    }), 200


if __name__ == "__main__":
    app.run(port=5000, debug=True)

dog running faster cat visit
 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
    ~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/traitlets/config/application.py", line 1074, in launch_instance
    app.initialize(argv)
    ~~~~~~~~~~~~~~^^^^^^
  File "/Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/traitlets/config/application.py", line 118, in inner
    return method(app, *args, **kwargs)
  File "/Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 692, in initialize
    self.init_sockets()
    ~~~~~~~~~~~~~~~~~^^
  File "/U

SystemExit: 1

/Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [3]:
import os

MODEL_DIR = "model_artifacts"
if not os.path.exists(MODEL_DIR):
    os.makedirs(MODEL_DIR)

In [2]:
import joblib

joblib.dump(vectorizer, f"{MODEL_DIR}/vectorizer.joblib")
joblib.dump(model, f"{MODEL_DIR}/model.joblib")

['model_artifacts/model.joblib']

In [4]:
import streamlit as st
import joblib
import os

# Local imports from your project
from Scrapper import extract_article_text
from preprocess import clean_text
from logger_db import init_db, log_query

# --- SETUP & MODEL LOADING ---

# Model paths
MODEL_DIR = "model_artifacts"
VECT_PATH = os.path.join(MODEL_DIR, "vectorizer.joblib")
MODEL_PATH = os.path.join(MODEL_DIR, "model.joblib")

# Use st.cache_resource to load the models only once
@st.cache_resource
def load_model_and_vectorizer():
    """Loads and returns the ML model and vectorizer."""
    if not os.path.exists(VECT_PATH) or not os.path.exists(MODEL_PATH):
        st.error("Model artifacts not found. Run train.py first.")
        st.stop()
    
    vectorizer = joblib.load(VECT_PATH)
    model = joblib.load(MODEL_PATH)
    return vectorizer, model

# Load the resources
vectorizer, model = load_model_and_vectorizer()

# Init DB (optional logging)
init_db()


# --- STREAMLIT UI ---

st.title("Factify: Fake News Detector")
st.write("Enter the URL of a news article or paste the text directly to check if it's real or fake.")

# Create tabs for different input methods
tab1, tab2 = st.tabs(["🔗 Analyze by URL", "📝 Analyze by Text"])

with tab1:
    url_input = st.text_input("Enter the article URL:", key="url_input")
    analyze_url_button = st.button("Analyze URL", key="url_button")
    
    if analyze_url_button and url_input:
        with st.spinner("Extracting text and making a prediction..."):
            # Step 1: Extract text
            try:
                text = extract_article_text(url_input)
                if not text or len(text.strip()) < 100:
                    st.error("Could not extract sufficient text from the URL. Please try another one or paste the text directly.")
                    st.stop()
            except Exception as e:
                st.error(f"An error occurred while fetching the URL: {e}")
                st.stop()

            # Step 2: Preprocess and Predict
            cleaned = clean_text(text)
            X = vectorizer.transform([cleaned])
            proba = model.predict_proba(X)[0]
            pred_index = proba.argmax()
            pred_class = model.classes_[pred_index]
            label = "Real" if pred_class == 1 else "Fake"
            confidence = float(proba[pred_index])

            # Step 3: Log and display results
            log_query(url_input, label, confidence, len(text))
            
            if label == "Fake":
                st.error(f"Result: This article is likely **{label}**")
            else:
                st.success(f"Result: This article is likely **{label}**")

            st.metric(label="Confidence", value=f"{confidence:.2%}")
            
            with st.expander("Show extracted text"):
                st.write(text)


with tab2:
    text_input = st.text_area("Paste the article text here:", height=250, key="text_input")
    analyze_text_button = st.button("Analyze Text", key="text_button")

    if analyze_text_button and text_input:
        with st.spinner("Making a prediction..."):
            # Step 1: Use the provided text
            text = text_input

            # Step 2: Preprocess and Predict
            cleaned = clean_text(text)
            X = vectorizer.transform([cleaned])
            proba = model.predict_proba(X)[0]
            pred_index = proba.argmax()
            pred_class = model.classes_[pred_index]
            label = "Real" if pred_class == 1 else "Fake"
            confidence = float(proba[pred_index])
            
            # Step 3: Log and display results
            log_query("RAW_TEXT", label, confidence, len(text))

            if label == "Fake":
                st.error(f"Result: This article is likely **{label}**")
            else:
                st.success(f"Result: This article is likely **{label}**")
                
            st.metric(label="Confidence", value=f"{confidence:.2%}")

2025-09-21 17:30:14.848 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-21 17:30:14.873 
  command:

    streamlit run /Users/mohansharma/Desktop/factify/backend-main/venv/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-09-21 17:30:14.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-21 17:30:14.873 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-21 17:30:14.874 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-21 17:30:14.906 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-21 17:30:14.907 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-